# <center>Deep Generative Models</center>
## <center>Seminar 12 — Video Diffusion: From Image to Video Generation</center>

<center><b>April 23, 2026</b></center>

**Plan**:
1. From image diffusion to video diffusion: temporal attention, 3D UNet vs DiT (15 min)
2. Architecture overview: Sora, Kling, CogVideoX, HunyuanVideo, Mochi, LTX, Wan, Veo (20 min)
3. Autoregressive video: next-frame prediction, causal attention, world models (15 min)
4. Practice: short video generation with Wan 2.1-1.3B / SVD / LTX (30 min)
5. Open problems: temporal consistency, long videos, motion control (7 min)
6. Summary and connection to the course (3 min)

> This seminar directly builds on **Seminar 11** (Evolution of Stable Diffusion). The main thesis: video diffusion in 2024–2025 recapitulates image diffusion's 2022–2023 evolution with a ~6-month lag — the same four axes (architecture, VAE, text encoder, objective) played out again, this time on 4D tensors.

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors sentencepiece protobuf
!pip install -q imageio imageio-ffmpeg mediapy ftfy
# For quantization (needed only if you try to squeeze LTX / CogVideoX-2B onto T4):
# !pip install -q optimum-quanto bitsandbytes

import torch
import gc
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import mediapy as media
from diffusers.utils import export_to_video, load_image
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

assert torch.cuda.is_available(), "GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

SEED = 42

## 1. From Image to Video Diffusion — Theory Bridge

This section connects everything we learned about image diffusion (Seminars 9–11, Lectures 11–12) with the video setting. We will see that **the four axes of evolution are the same** — they are just played out on one-dimension-larger tensors.

### 1.1 Why Video is Hard

A video is a **4D tensor**:
$$\mathbf{x} \in \mathbb{R}^{T \times H \times W \times C}$$

Three problems appear the moment we add the time axis:

1. **Naive per-frame generation is unusable.** Running a frozen SD 1.5 on every frame with the same seed gives a slideshow with flickering textures and morphing faces — nothing ties neighbouring frames together.

2. **Full 3D attention is infeasible.** An attention block over all spatio-temporal tokens costs $O((T \cdot H \cdot W)^2)$ — for a 5-second 720p clip that is $\sim 10^{11}$ attention entries per head, per layer.

3. **Data scarcity.** High-quality video-text pairs are 2–3 orders of magnitude less abundant than image-text pairs. Every design choice has to make video-native training cheap enough to reach scale.

Every recipe in this seminar is a response to at least one of these three constraints.

### 1.2 Temporal Attention — Factorized Space-Time (the VDM recipe)

The first successful design — **Video Diffusion Models** (Ho et al. 2022) — starts from an image U-Net and adds time by **factorizing** the attention.

**Step-by-step construction** (Ho et al. 2022):
- Take a 2D U-Net for images.
- Replace 2D convolutions (`3×3`) with **space-only 3D convolutions** (`1×3×3`). Frames are treated as a batch dimension — no information flows across time in convolutions.
- Insert a **temporal attention block** after each spatial attention block.

The two blocks differ only in how the batch and sequence axes are arranged:

```
   Spatial block                     Temporal block
   (frames are the batch)            (pixels are the batch)
  ┌──────────────────────┐          ┌──────────────────────┐
  │ batch = (B, T)       │          │ batch = (B, H·W)     │
  │ sequence = H·W       │   ───►   │ sequence = T         │
  │ cost per (B·T) item  │          │ cost per (B·N) item  │
  │ = N²                 │          │ = T²                 │
  └──────────────────────┘          └──────────────────────┘
```

Total attention cost with $N = H \cdot W$:

$$\underbrace{O(N^2 T)}_{\text{spatial}} + \underbrace{O(N T^2)}_{\text{temporal}} \ll \underbrace{O(N^2 T^2)}_{\text{full 3D}}$$

[**Accent**] If temporal attention is disabled, the model collapses to an independent image model over frames. This lets us **train jointly on images + videos** — since high-quality video-text pairs are scarce but image-text pairs are abundant, this trick was the unlock for every 2022–2023 video model.

### 1.3 Family Tree — Temporal Layers in Image Models

The same factorized recipe powers the pre-transformer generation:

| Model | Year | Trick |
|---|---|---|
| **VDM** (Ho et al.) | 2022 | Canonical recipe: factorized space-time 3D U-Net |
| **Imagen Video** (Google) | 2022 | Cascade of **7** models: base + 3 spatial SR + 3 temporal SR |
| **Make-A-Video** (Meta) | 2022 | Temporal layers stacked on a **frozen** T2I backbone — no paired video-text needed |
| **Video LDM / Align Your Latents** (NVIDIA) | 2023 | Insert temporal conv + attn into a pretrained image LDM; train only the new layers |
| **Stable Video Diffusion** | 2023 | Same recipe, fine-tune everything; 14–25 frames at 576×1024. Our live-demo baseline. |
| **Lumiere** (Google) | 2024 | **Space-Time U-Net** generating the whole video in one pass (not a cascade) |

All of these stay in the **U-Net + ε-prediction** regime — the image side of the field had not yet made the transformer jump.

> **References**: Ho et al. "Video Diffusion Models" arXiv:2204.03458 · Ho et al. "Imagen Video" 2022 · Singer et al. "Make-A-Video" 2022 · Blattmann et al. "Align Your Latents / Video LDM" 2023 · Blattmann et al. "Stable Video Diffusion" 2023 · Bar-Tal et al. "Lumiere" 2024

### 1.4 The Paradigm Shift — DiT for Video

We saw in Seminar 11 that images went `UNet → DiT → MMDiT → FLUX`. Video follows the same trajectory, about six months behind. The translation is straightforward: reuse the image DiT block, but feed it **spacetime patches** instead of image patches.

**Patchification in 4D**. A latent video $\mathbf{z} \in \mathbb{R}^{T' \times H' \times W' \times C}$ is cut into **tubelets** of shape $p_t \times p_h \times p_w$ (e.g. $1 \times 2 \times 2$ or $2 \times 2 \times 2$):

$$\mathbf{z} \;\xrightarrow{\text{patch embed}}\; \mathbf{s} \in \mathbb{R}^{N \times D},\qquad
N = \frac{T' H' W'}{p_t p_h p_w}$$

This single change has a big consequence: because the transformer only sees a **sequence of tokens**, the model becomes **resolution- and duration-agnostic** — an `N=1024` sequence can be $8\times32\times32$ just as easily as $16\times16\times32$. Sora's variable-resolution, variable-duration inference is exactly this property being exploited.

```
  Pixel video         Latent video           Token sequence
  (T, H, W, 3)   ──►  (T',H',W',Cz)   ──►    (N, D)   ──►   [ DiT ]
   3D Causal VAE       patchify             transformer
```

### 1.5 The 3D Causal VAE — The New Primitive

The 2D image VAE from SD is no longer sufficient: we need to compress both space and time, and we want the encoder to be **streaming-friendly** so that we can extend / stitch long videos.

**Building block — CausalConv3D**. A 3D convolution with padding applied **only in the past direction** along the time axis. Every output frame depends only on current + previous input frames, never on future frames — the causal property.

**Typical compression**: `8× spatial × 8× spatial × 4× temporal`. A 720×1280×129 video (3 channels) compresses as:
$$(T, H, W, C) = (129, 720, 1280, 3) \;\xrightarrow{\text{VAE}}\; (T', H', W', C_z) = (33, 90, 160, 16)$$

which is $\approx 185 \times$ fewer elements — enough to make a 3D transformer tractable.

**Latent shape in general**:
$$\mathbf{z} \in \mathbb{R}^{T/4 \,\times\, H/8 \,\times\, W/8 \,\times\, C_z}, \qquad C_z \in \{4, 16\}$$

**Tiling for long videos**. For videos longer than the VAE was trained on, encode and decode in **overlapping spatial + temporal tiles** and blend at the seams. Without tiling the 3D VAE alone exhausts VRAM long before the transformer does.

Anti-flicker loss. Modern 3D VAEs (CogVideoX, HunyuanVideo) are trained with explicit **temporal smoothness** losses on the reconstruction — this is the first line of defence against per-frame flickering.

### 1.6 3D RoPE — Positional Encoding for Spacetime

Positional encoding must generalise across resolutions, aspect ratios, and video lengths. Absolute / learned PE is a bad fit — RoPE (Seminar 11) solves this for images. The extension to video is straightforward: split the head dimension and rotate each slice independently per axis.

**Construction**. Decompose the per-head embedding dimension:
$$d = d_t + d_h + d_w$$
and for a token at position $(m_t, m_h, m_w)$ apply **three independent RoPE rotations**:
$$\text{RoPE}_t(m_t) \;\oplus\; \text{RoPE}_h(m_h) \;\oplus\; \text{RoPE}_w(m_w)$$

Each sub-slice encodes **relative** distance along one axis. The whole scheme is permutation-equivariant in a useful way: rotating the video in time or shifting it in space leaves attention scores invariant up to the expected relative offset.

[**Accent**] 3D RoPE is the default in CogVideoX, HunyuanVideo, LTX, and Wan. It is the single encoding change that makes generating at multiple resolutions/lengths work reliably.

> **Reference**: VideoRoPE arXiv:2502.05173

---
*With factorized / joint space-time attention, 3D causal VAE, and 3D RoPE as our primitives, we can now look at concrete modern architectures.*

## 2. Architecture Overview — Sora, Kling, CogVideoX, HunyuanVideo, Mochi, LTX, Wan, Veo

We walk the 2024–2025 landscape. For each model the question is always the same three-slot template: **what backbone · what VAE · what text encoder · what objective**. The answers tell a very consistent story.

### 2.1 Sora — OpenAI (Feb 2024, closed)

The model that started the modern video race. Only a technical report was released, but the key design choices are documented.

- **Backbone**: DiT on spacetime patches (no factorization — full joint attention).
- **VAE**: a learned **spatiotemporal** autoencoder trained from scratch — not the SD image VAE.
- **Positional encoding**: 3D absolute PE over $(t, h, w)$ — allows **variable resolution, duration, and aspect ratio** as direct inputs.
- **Parameter count**: undisclosed; community estimates $\sim 3{-}10$B.
- **Training trick**: "DALL-E 3 re-captioning" — a dense captioner re-labels every training video, massively boosting prompt adherence.

```
Sora (DiT on Spacetime Patches):

  Video ─[Spatiotemporal VAE]─► Latent (T',H',W',C)
              │
              ▼  patchify into tubelets (e.g. 2×2×2)
        ┌──────────────────────────┐
        │  Sequence of N tokens    │
        │  with 3D positional enc. │
        └────────────┬─────────────┘
                     ▼
                [  DiT blocks  ]
                     │
                     ▼
        [Spatiotemporal VAE Decoder] → Video
```

[**Accent**] Takeaway: "DiT on spacetime patches with variable input shape" is the modern recipe. Every other DiT-based video model below is a variation of Sora.

> **Reference**: Brooks et al. "Video generation models as world simulators" OpenAI tech report, Feb 2024

### 2.2 Kling — Kuaishou (June 2024, closed API)

- **Backbone**: DiT with **full spatiotemporal self-attention** (no factorization).
- **VAE**: in-house 3D VAE.
- **Notable spec**: up to **2-minute clips at 1080p, 30 fps** — the longest among closed models.
- No architectural paper: the design is reconstructed from product blog posts and interviews.

Kling's existence shows that **full 3D attention** is feasible at scale given enough compute; the open community responded with sparse / factorized attention (see §5.2).

### 2.3 CogVideoX — Tsinghua / Zhipu (Aug 2024, ICLR 2025, OPEN)

Our **main case study**: the best open-source DiT-based video model with full architectural details in the paper.

- **Variants**: `CogVideoX-2B`, `CogVideoX-5B`, `CogVideoX1.5-5B`, plus I2V variants.
- **Output**: 6–10 s videos at 16 fps, up to 768×1360 (5B) / 720×480 (2B).
- **Text encoder**: T5-v1.1-xxl (4096-dim) — same T5 branch as SD 3.

**Three core innovations** — each gets its own slide below.

> **Reference**: Yang et al. "CogVideoX: Text-to-Video Diffusion Models with an Expert Transformer", arXiv:2408.06072, ICLR 2025. HF IDs: `zai-org/CogVideoX-{2b,5b,5b-I2V}`.

#### 2.3.1 CogVideoX Innovation 1 — 3D Causal VAE with $8 \times 8 \times 4$ Compression

- CausalConv3D stack, spatial downsample $\times 8$ per side, temporal downsample $\times 4$.
- Trained with **explicit anti-flicker reconstruction losses** (consecutive-frame L1 and feature-matching) — this is where "the cat's fur does not shimmer" comes from.
- 16 latent channels (more than SD's 4) → tighter reconstruction, easier for the transformer.
- Supports **tiling** at inference for longer clips than the training length.

#### 2.3.2 CogVideoX Innovation 2 — Expert MMDiT + Expert-AdaLN (the key novelty)

Recall MMDiT from Seminar 11: text and video tokens are concatenated and passed through **joint self-attention**, with separate Q/K/V/FFN per modality. CogVideoX keeps all of that. The new piece is that the **AdaLN timestep modulation is now also per-modality**:

$$\text{AdaLN}_{\text{video}}(\mathbf{h}_v,\, \mathbf{c}) = \gamma_v(\mathbf{c})\, \text{LN}(\mathbf{h}_v) + \beta_v(\mathbf{c})$$
$$\text{AdaLN}_{\text{text}}(\mathbf{h}_t,\, \mathbf{c}) = \gamma_t(\mathbf{c})\, \text{LN}(\mathbf{h}_t) + \beta_t(\mathbf{c})$$

with separate MLPs producing $(\gamma_v, \beta_v)$ and $(\gamma_t, \beta_t)$ from the shared conditioning vector $\mathbf{c}$ (timestep embedding).

[**Accent**] Contrast with SD 3: in SD3-MMDiT a **single** AdaLN modulates both streams. Here each modality has its **own "expert" modulation** — hence "Expert MMDiT". Since video and text statistics (scale, sparsity, temporal structure) are very different, this per-modality conditioning is measurably better than shared AdaLN in the ablations.

```
   video tokens z_v           text tokens z_t
         │                          │
   AdaLN_video(c)            AdaLN_text(c)      ◄── per-modality experts
         │                          │
    Q_v, K_v, V_v            Q_t, K_t, V_t
         │                          │
         └──────── concat ──────────┘
                     ▼
              JOINT SELF-ATTN
                     ▼
         split ──────┬──────
         │                  │
       FFN_v              FFN_t
```

#### 2.3.3 CogVideoX Innovation 3 — 3D RoPE

Same 3D RoPE as §1.6 — independent rotary embeddings on $(t, h, w)$. In CogVideoX ablations this converges **substantially faster** than sinusoidal absolute PE and generalises better to unseen resolutions / lengths.

**Training recipe**: progressive low-res → high-res, multi-resolution frame packs, classifier-free guidance on the text prompt, Rectified-Flow / v-prediction objective.

### 2.4 HunyuanVideo — Tencent (Dec 2024, OPEN, 13B)

Too big to run on T4 (needs $\geq$14 GB in FP8 plus an 8B text encoder), but architecturally important: it brings the **FLUX design** to video.

Three components:

1. **3D causal VAE**: CausalConv3D, $8\times$ spatial, $4\times$ temporal, 16 channels — the same template as CogVideoX.

2. **MLLM text encoder** — `xtuner/llava-llama-3-8b-v1_1` decoder-only LLM + a bidirectional **token refiner**. This **replaces T5/CLIP entirely**. Claimed to beat both on image-text alignment; the decoder-only LLM gives richer instruction-following semantics.

3. **Dual-stream → single-stream DiT** (exactly the FLUX pattern from Seminar 11):
   - **Early blocks (dual-stream)**: separate Q/K/V and MLP per modality, interact only through joint attention.
   - **Late blocks (single-stream)**: concatenate into one sequence with shared weights.

**Objective**: Flow Matching (v-prediction) — same shift as SD 1.5 → SD 3 for images.

[**Accent**] Translation table to Seminar 11:

| Image side (Seminar 11) | HunyuanVideo counterpart |
|---|---|
| FLUX dual+single stream | HunyuanVideo dual+single stream |
| Triple encoder (CLIP + OpenCLIP + T5) | **MLLM (llava-llama-3-8b)** |
| Rectified Flow v-prediction | Flow Matching v-prediction |

> **Reference**: Kong et al. "HunyuanVideo: A Systematic Framework for Large Video Generation Models", Tencent, Dec 2024.

### 2.5 Mochi-1 — Genmo (Oct 2024, OPEN, 10B)

- **AsymmDiT** (Asymmetric Diffusion Transformer): the visual stream has $\sim 4\times$ more parameters than the text stream.
- **Non-square QKV and output projections** unify the two streams at attention time.
- Memory-efficient at inference vs symmetric MMDiT for the same visual capacity.
- **Single T5-XXL** text encoder (no MLLM).
- **Full 3D attention** over **44,520 video tokens** — excellent quality, but prohibitive for long clips.
- **Output**: 480p, 5.4 s @ 30 fps.

Mochi's design statement: "most of the work happens on the visual side, so spend the parameters there".

### 2.6 LTX-Video — Lightricks (Dec 2024, OPEN, arXiv:2501.00103)

The **fastest open video model**. Two related ideas:

- **Extreme 1:192 compression VAE**: a single latent token represents a $32 \times 32 \times 8$ pixel tubelet. Two orders of magnitude more aggressive than the usual $8\times 8\times 4$.
- **Patchification moved from transformer into the VAE itself**. The VAE outputs tokens directly — no tokenization layer in the DiT.
- **Decoder performs the final denoising step** — the line between VAE and diffusion is blurred.

**Speed**:
- 5 s @ 24 fps @ 768×512 in **2 s on an H100**.
- 121 frames in 11 s on an RTX 4090.

**VRAM**: $\sim$10 GB nominal, down to $\sim$6 GB with Q8 quantisation — **T4-possible**.

> **Reference**: HaCohen et al. "LTX-Video: Realtime Video Latent Diffusion", arXiv:2501.00103.

### 2.7 Wan 2.1 / Wan 2.2 — Alibaba (2025, OPEN, Apache 2.0)

Alibaba's open release with a consumer-grade variant and a next-gen MoE variant.

**Wan 2.1**:
- `Wan2.1-T2V-1.3B` — $\sim 8.2$ GB VRAM in bf16 → the **only model in this seminar that actually fits free Colab T4**. This is our main live demo.
- `Wan2.1-T2V-14B` — theory only for us.
- Standard DiT + cross-attention, 3D VAE, T5 text encoder, Flow Matching objective.

**Wan 2.2 — MoE along the denoising trajectory** (the new idea):

```
  noise level   │ high    ├───── switch point ─────┤   low
  time  t       │ 0                                     1
                │
  expert used   │   HIGH-NOISE EXPERT   │    LOW-NOISE EXPERT
                │      (14B params)     │      (14B params)
                │                       │
  handles       │  layout, composition  │  detail, texture
```

- Two $\sim$14B experts, **switched mid-rollout** by signal-to-noise ratio (when the noise proportion drops by half).
- Total 27B params; only 14B are **active** per step — memory/compute parity with a 14B dense model.
- Training objective: Flow Matching.

[**Accent**] Observation from Seminar 11 carried over: image MoE inside a layer (Mixture-of-Experts transformer) never really caught on; **MoE along the noise axis** is a new axis that only makes sense for diffusion/FM.

### 2.8 Veo 3 — Google DeepMind (2025, closed API)

- **Latent Diffusion Transformer** jointly generating **video + audio**.
- Diffusion applied to a **unified token sequence** interleaving spatiotemporal video latents with temporal audio latents — a single attention stack synchronises lips, footsteps, and ambience to pixels.
- Sub-components (community estimates): 12B transformer producing keyframes every 2 s, a 28B U-Net interpolator between keyframes, and a 9B audio engine.
- Reported lip-sync accuracy **< 120 ms**.

Veo 3 is the first model to make "joint A/V generation" a first-class design goal, not an afterthought.

> **Reference**: DeepMind "Veo 3 technical report", 2025.

### 2.9 The Summary Table

| Model | Year | Params | Backbone | VAE | Text encoder | Objective | Key innovation |
|---|---|---|---|---|---|---|---|
| VDM | 2022 | $<1$B | 3D U-Net | pixel | BERT-style | DDPM $\epsilon$ | Factorized space-time attn |
| SVD | 2023 | $\sim1.5$B | 2D + temporal U-Net | SD VAE | CLIP | DDPM | Temporal layers in LDM |
| Lumiere | 2024 | $\sim3$B | Space-Time U-Net | pixel | T5 | DDPM | Single-pass full-video |
| **Sora** | 2024 | $\sim3{-}10$B | DiT | in-house 3D | T5-like | DDPM-era | Spacetime patches, variable shape |
| **Kling** | 2024 | — | DiT + 3D VAE | in-house | — | — | 2-min 1080p (full 3D attn) |
| **CogVideoX** | 2024 | 2B / 5B | **Expert MMDiT** | 3D causal $8\times8\times4$ | T5-XXL | v-pred | Per-modality AdaLN, 3D RoPE |
| **Mochi-1** | 2024 | 10B | **AsymmDiT** | 3D VAE | T5-XXL | Flow Matching | Asymmetric 4× visual stream |
| **HunyuanVideo** | 2024 | 13B | **Dual→Single DiT** | 3D causal | **MLLM (llava-llama-3-8b)** | Flow Matching | FLUX blocks + MLLM text |
| **LTX-Video** | 2024 | 2B / 13B | DiT | **1:192 VAE** | T5-XXL | Flow Matching | Patchify inside VAE |
| **Wan 2.1** | 2025 | 1.3B / 14B | DiT + cross-attn | Wan-VAE | T5 | Flow Matching | Consumer-grade 1.3B |
| **Wan 2.2** | 2025 | 27B (14B active) | **DiT MoE along noise** | Wan-VAE | T5 | Flow Matching | High/low-noise experts |
| **Veo 3** | 2025 | 12B+28B+9B | LDT | spatiotemporal + audio | — | — | **Joint audio-video** |

[**Important — three overarching narratives**]:

1. **Architecture**: Factorized U-Net $\to$ full-3D U-Net $\to$ DiT (Sora) $\to$ MMDiT (CogVideoX) $\to$ Dual+Single-stream (HunyuanVideo) $\to$ MoE-along-noise (Wan 2.2). **Exact parallel to the image evolution `UNet → DiT → MMDiT → FLUX` from Seminar 11.**

2. **VAE compression**: pixel $\to$ 2D image VAE (SVD) $\to$ 3D causal VAE $8\times 8\times 4$ (everyone modern) $\to$ extreme 1:192 (LTX).

3. **Training objective**: DDPM $\epsilon$-prediction $\to$ Flow Matching v-prediction. Video followed image with a ~6-month lag.

## 3. Autoregressive Video — Next-Frame Prediction and World Models

So far every model we have seen is a **parallel diffusion** model: the whole clip is generated from noise at once. There is a second paradigm — **autoregressive video** — where frames (or blocks of frames) are generated one at a time, conditioning on the past. This is the paradigm of LLMs lifted to video.

### 3.1 Two Paradigms Side by Side

| Aspect | **Parallel diffusion** | **Autoregressive** |
|---|---|---|
| Generation | All frames together | One frame (or block) at a time |
| Attention mask in time | Bidirectional | Causal |
| Clip length | Fixed (training budget) | Arbitrary, streaming |
| Error accumulation | None within a clip | Compounds over rollout |
| Temporal consistency | Strong on short clips | Weak on long horizons |
| Natural use-cases | Short high-quality clips | World models, games, streaming |
| Examples | Sora, CogVideoX, HunyuanVideo, LTX | VideoGPT, MAGVIT-v2, Genie 2, GameNGen |

The two regimes are converging (see Genie 2 below), but the paradigmatic difference is worth understanding on its own.

### 3.2 Causal Attention in Time

The only architectural change needed to go from parallel → autoregressive is an **attention mask**. Within a single frame the model can attend freely in space; across frames, the mask is **causal**:

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}} + M\right)V$$

$$M_{ij} = \begin{cases} 0 & \text{if }\text{frame}(i) \geq \text{frame}(j) \\ -\infty & \text{otherwise} \end{cases}$$

This is exactly the decoder-LLM causal mask, lifted to a 3D spacetime token grid. A token at frame $t$ can see all tokens from frames $\leq t$ (including its own spatial neighbours) but never any token from frame $> t$.

### 3.3 VideoGPT — Yan et al. 2021 (the canonical baseline)

The blueprint that everything autoregressive builds on.

- **Stage 1** — 3D VQ-VAE with 3D convolutions + axial self-attention learns **discrete latent tokens** for video clips.
- **Stage 2** — a GPT-style transformer autoregressively models the token sequence with spatiotemporal position encoding.
- Competitive with GANs on BAIR and UCF-101 — the first strong "video = tokens + language model" result.

> **Reference**: Yan et al. "VideoGPT: Video Generation using VQ-VAE and Transformers", arXiv:2104.10157.

### 3.4 MAGVIT-v2 — Yu et al. ICLR 2024: "Language Model Beats Diffusion"

A landmark: a pure LLM over visual tokens **beats diffusion** on ImageNet and Kinetics.

Two key ideas:
- A **joint image + video tokenizer** — the same VQ vocabulary for still images and videos.
- **Lookup-Free Quantization (LFQ)** — each latent dimension is independently binarised, allowing the codebook to scale to $2^{18}$ entries, previously infeasible.

Punchline from the paper title: **"The tokenizer is key."** Given a good enough discrete video tokenizer, a plain autoregressive LLM is competitive with (or better than) a diffusion model.

> **Reference**: Yu et al. "Language Model Beats Diffusion — Tokenizer is Key for Visual Generation", arXiv:2310.05737, ICLR 2024.

### 3.5 Genie 2 — DeepMind (Dec 2024): autoregressive latent **diffusion** world model

Genie 2 is the hybrid. It is **autoregressive across frames** but **diffusion within each frame**:

```
  latent frame_1 ──► latent frame_2 ──► latent frame_3 ──► ...
       ▲                 ▲                   ▲
       │                 │                   │
  diffusion over    same, conditioned   same, conditioned
  the latent        on previous         on all previous
                    frame + action      frames + actions
```

- Each latent frame is **denoised by diffusion** (not quantised) — hybrid approach.
- Latent frames are fed autoregressively to a large **causal-masked dynamics transformer** — like an LLM whose tokens are whole diffusion-generated frames.
- **Classifier-free guidance on actions** → controllable simulation.
- Generates consistent **10–60-second 3D worlds** from a single prompt.

Genie 2 is the current clearest example of the diffusion × autoregressive merge.

### 3.6 GameNGen — Valevski et al. ICLR 2025: real-time game engine

- Fine-tuned **Stable Diffusion 1.4** as a **next-frame predictor** conditioned on the user's action + the last $N$ frames.
- Simulates **DOOM at 20 fps** on a single TPU.
- Image quality indistinguishable from the real engine in user studies.
- An RL agent is used to generate 10M env-step training trajectories.
- Limitation: memory window is $\sim 3$ s — architectural bottleneck for long-horizon consistency.

> **Reference**: Valevski et al. "Diffusion Models Are Real-Time Game Engines", arXiv:2408.14837, ICLR 2025.

### 3.7 Summary: Parallel vs Autoregressive — When to Use Which

**Parallel diffusion** wins when you want:
- A short, highly polished clip (e.g. a 5-s ad).
- Strong global temporal coherence.
- Maximum sample quality per compute.

**Autoregressive** (possibly with diffusion inside each step) wins when you want:
- Streaming / unbounded length.
- **Interactivity** — condition on user actions, run a game engine, simulate an environment.
- Compute on demand (one frame at a time) instead of one big inference.

[**Transition**]: *Now let's actually run a video model.*

## 4. Practice — Generating Videos

We run three models that actually fit on free-Colab T4 (16 GB VRAM):

1. **Stable Video Diffusion** — the 2023 3D-U-Net + temporal-layers recipe (image-to-video).
2. **Wan 2.1 T2V-1.3B** — our main text-to-video demo (Apache 2.0, $\sim$8 GB bf16).
3. **LTX-Video** — the speed baseline (image-to-video) with memory offload.

Between every load we clean GPU state — this matters on T4.

### 4.1 Demo 1 — Stable Video Diffusion (historical 3D-U-Net, Image-to-Video)

This is the 2023 recipe: a frozen image LDM (Stable Diffusion 2.1) plus inserted temporal conv + attention layers, fine-tuned on a curated video dataset. It represents **everything that came before the DiT-for-video shift**.

In [ ]:
from diffusers import StableVideoDiffusionPipeline

pipe_svd = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

# Memory-friendly: offload text encoder / VAE when idle
pipe_svd.enable_model_cpu_offload()

print(f"SVD U-Net params: {sum(p.numel() for p in pipe_svd.unet.parameters())/1e6:.0f}M")

In [ ]:
# Use the rocket reference image from HF docs
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/diffusers/svd/rocket.png"
)
image = image.resize((1024, 576))

generator = torch.Generator("cuda").manual_seed(SEED)
frames = pipe_svd(
    image,
    decode_chunk_size=8,       # decode 8 frames at a time to save VRAM
    generator=generator,
    num_frames=14,
    motion_bucket_id=127,      # higher = more motion
    noise_aug_strength=0.02,
).frames[0]

export_to_video(frames, "svd_rocket.mp4", fps=7)
media.show_video(media.read_video("svd_rocket.mp4"), fps=7)

In [ ]:
# Cleanup — always between model loads
del pipe_svd
torch.cuda.empty_cache()
gc.collect()
print(f"After SVD cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

**What to notice in the SVD output**:
- Clear *camera* motion (this is what the temporal layers primarily learned).
- Very limited *scene* dynamics — the rocket does not light, exhaust does not evolve, because the temporal layers were bolted onto a frozen image model and the model has no strong world-dynamics prior.
- This is the **characteristic artefact of the 3D-U-Net generation**. Modern DiT-based models (Wan, CogVideoX, Hunyuan) look qualitatively different — richer internal motion, fewer "slideshow" moments.

### 4.2 Demo 2 — Wan 2.1 T2V-1.3B (main live demo)

`Wan2.1-T2V-1.3B` is the centrepiece of this seminar: it is the **only model in our list that truly fits free Colab T4** (~8 GB bf16). Its design — DiT + cross-attention + 3D VAE + T5 + Flow Matching — is the mainstream open-source consensus of 2025.

In [ ]:
from diffusers import WanPipeline, AutoencoderKLWan

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

# VAE stays in fp32 for numerical stability (per HF docs recommendation)
vae = AutoencoderKLWan.from_pretrained(
    model_id, subfolder="vae", torch_dtype=torch.float32
)
pipe_wan = WanPipeline.from_pretrained(
    model_id, vae=vae, torch_dtype=torch.bfloat16
)
pipe_wan.to("cuda")

print(f"Wan 2.1 transformer params: "
      f"{sum(p.numel() for p in pipe_wan.transformer.parameters())/1e9:.2f}B")

In [ ]:
prompt = (
    "A cat wearing a wizard hat casts a glowing rainbow spell in a dusty "
    "medieval library, cinematic lighting, highly detailed"
)
negative = "blurry, low quality, distorted, static, text, watermark"

generator = torch.Generator("cuda").manual_seed(SEED)
result = pipe_wan(
    prompt=prompt,
    negative_prompt=negative,
    height=480,
    width=832,
    num_frames=81,                # ~5 s at 16 fps
    num_inference_steps=30,
    guidance_scale=5.0,           # lower than SD 1.5 — typical for video FM
    generator=generator,
).frames[0]

export_to_video(result, "wan_cat.mp4", fps=16)
media.show_video(media.read_video("wan_cat.mp4"), fps=16)

**Observations**:
- Character identity is **stable** across all 81 frames — no face morphing. This is the 3D causal VAE with anti-flicker training doing its job.
- Rich internal dynamics: the hat moves with the head, the spell evolves, the camera drifts. Compare with SVD above.
- The prompt is followed well despite T5-only text conditioning — Wan's training data and captioning pipeline are strong.

### 4.3 Demo 3 — Parameter Exploration

With the same pipeline loaded, sweep the main knobs and observe the trade-offs.

In [ ]:
# Fixed prompt / seed, vary inference steps
prompt_fixed = "A drone shot flying over a foggy mountain ridge at sunrise, cinematic"
negative_fixed = "blurry, low quality, static"

sweeps = [("steps=15", dict(num_inference_steps=15, guidance_scale=5.0)),
          ("steps=30", dict(num_inference_steps=30, guidance_scale=5.0)),
          ("cfg=3.0",  dict(num_inference_steps=30, guidance_scale=3.0)),
          ("cfg=7.5",  dict(num_inference_steps=30, guidance_scale=7.5))]

videos = {}
for name, kwargs in sweeps:
    generator = torch.Generator("cuda").manual_seed(SEED)
    out = pipe_wan(
        prompt=prompt_fixed,
        negative_prompt=negative_fixed,
        height=480, width=832,
        num_frames=33,                  # shorter for the sweep — ~2 s
        generator=generator,
        **kwargs,
    ).frames[0]
    path = f"wan_sweep_{name}.mp4"
    export_to_video(out, path, fps=16)
    videos[name] = path
    print(f"saved {name} → {path}")

In [ ]:
# Display the sweep side by side
for name, path in videos.items():
    print(name)
    media.show_video(media.read_video(path), fps=16)

**What to look for**:
- **Steps 15 → 30**: structure is roughly formed by step 15 (Flow Matching does a lot in few steps), but fine textures and motion coherence improve with more steps.
- **CFG 3.0 vs 7.5**: higher guidance → tighter prompt adherence but more saturation / artefacts. For video the sweet spot is **typically 4–6**, noticeably lower than the 7–9 range common for SD 1.5 images.

### 4.4 Demo 4 — Image-to-Video with LTX-Video

Let's now swap to LTX-Video — the 1:192-compression VAE model — to get a feel for its speed advantage. We will use its I2V pipeline.

In [ ]:
# Free Wan before loading LTX (both won't fit together)
del pipe_wan, vae
torch.cuda.empty_cache()
gc.collect()
print(f"After Wan cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

In [ ]:
from diffusers import LTXImageToVideoPipeline

pipe_ltx = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video", torch_dtype=torch.bfloat16
)
pipe_ltx.enable_model_cpu_offload()  # keep peak VRAM low

# A still image to animate — reuse the SVD rocket, different resolution
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/diffusers/svd/rocket.png"
).resize((768, 512))

prompt = "The rocket ignites its main engine and begins to lift off, clouds of exhaust billow outwards, cinematic"
neg = "blurry, low quality, static, no motion"

generator = torch.Generator("cuda").manual_seed(SEED)
frames = pipe_ltx(
    image=image,
    prompt=prompt,
    negative_prompt=neg,
    num_frames=97,                 # ~4 s at 24 fps
    num_inference_steps=40,
    guidance_scale=3.0,            # LTX wants lower CFG than Wan
    generator=generator,
).frames[0]

export_to_video(frames, "ltx_rocket.mp4", fps=24)
media.show_video(media.read_video("ltx_rocket.mp4"), fps=24)

In [ ]:
del pipe_ltx
torch.cuda.empty_cache()
gc.collect()
print(f"After LTX cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

### 4.5 Practical Patterns — What to Remember

- **Seed reuse** (`torch.Generator("cuda").manual_seed(SEED)`) is the only way to ablate a single parameter fairly — always reset the generator before each call.
- **Guidance scale sweet spot is lower for video**: typically **4–6**, not 7–9 as in SD 1.5. Higher CFG over-saturates colours and introduces motion artefacts.
- **Number of frames is quantised**. Most open models are trained around 49 / 81 / 129 frames; going substantially longer degrades quality (attention is $O(T^2)$ — the network was never trained on that shape).
- **`fps` is a display / export choice**, not a model parameter — the model produces frames at a fixed temporal rate set by the VAE's temporal stride. `fps=16` vs `fps=24` is a post-hoc decision about playback speed.
- **VAE in float32** for stability (Wan docs); **transformer in bfloat16**. Mixing precisions like this is standard for video pipelines on consumer GPUs.

### 4.6 Memory Management Recap

In [ ]:
def free_gpu(label=""):
    """Call after every `del pipe_X` to confirm memory was actually released."""
    torch.cuda.empty_cache()
    gc.collect()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"[{label}] allocated={allocated:.2f} GB, reserved={reserved:.2f} GB")

free_gpu("end of §4")

## 5. Open Problems — What's Still Broken in 2026

Generation quality has shot up, but four concrete failure modes keep video diffusion from being "solved".

### 5.1 Temporal Consistency

Even in state-of-the-art models you see:
- **Object identity drift** — a character's face subtly morphs over a 5-s clip.
- **Flickering fine textures** — fur, foliage, small patterns change frame-to-frame.
- **Physics violations** — objects pass through walls, gravity flips, water flows uphill.

**Root cause**: per-frame noise is sampled independently during training. Temporal correlation has to be *learned* — it is not structurally enforced.

**Current mitigations**:
- 3D causal VAEs trained with temporal-smoothness reconstruction losses (CogVideoX).
- Joint space-time attention (no factorization) — more expensive but structurally tighter (Kling, Mochi).
- Bigger models with more diverse training data.

### 5.2 The Long-Video Wall

Full 3D attention cost is **quadratic in the number of spatio-temporal tokens**:
$$\text{cost} \;=\; O\bigl((T \cdot H \cdot W)^2\bigr)$$

For 5 s at 720p with a HunyuanVideo-class model, **attention alone is $\sim$800 of 950 seconds** of inference time. Most open models cap at $\sim 5{-}10$ s clips. Going longer requires sparsity.

**Active research directions**:

| Method | Idea | Reference |
|---|---|---|
| **Sliding Tile Attention (STA)** | Hardware-aware tile-local attention in time/space | arXiv:2502.04507 |
| **FreeSwim** | Inward sliding window preserving the training receptive field | 2025 |
| **Mixture of Contexts (MoC)** | Sparse retrieval of relevant chunks + anchors per query | arXiv:2508.21058 |
| **Block-autoregressive rollouts** | Generate in blocks of frames, condition on previous blocks; self-forcing distillation | 2024–25 |
| **Lumiere-style** | Run temporal attention only at the coarsest space-time scale of a pyramid | 2024 |

[**Accent**] This is the **single biggest bottleneck** in the field right now. Every open question — long narratives, interactive simulation, high-fps generation — eventually bottoms out on this one.

### 5.3 Motion Control

Describing motion in text is fundamentally lossy. Current control tracks:

- **Text-based** ("the camera pans left") — unreliable, imprecise.
- **Trajectory-based** — draw curves for object motion (DragNUWA, MotionCtrl).
- **Camera control** — explicit camera pose inputs (CameraCtrl, MotionDirector).
- **Keyframe conditioning** — specify start / middle / end frames.
- **Physics priors** — Genie 2 learns **latent actions** from gameplay data; these give surprisingly crisp control.

### 5.4 Beyond Generation — Controllable World Models

Pure-quality generation is now close to saturation on standard benchmarks. The next frontier is **controllable simulation**:

- **Interactive world models** — video conditioned on a user action stream (Genie 2, GameNGen).
- **Applications**: RL environment simulation, game engines, robotics sim-to-real, virtual production.
- **Open question**: can we train a single foundation model that is both a good generator *and* a good simulator?

### 5.5 Current Frontier (early 2026)

- **Joint audio-video generation** (Veo 3, Wan-Audio): unified token sequence for pixels and waveforms.
- **Real-time generation**: LTX-Video and Wan 2.1-1.3B are approaching faster-than-real-time on a single 4090.
- **Consumer-grade models**: HunyuanVideo-1.5 (8.3B) explicitly targets $\leq 16$ GB GPUs.
- **MoE along the denoising trajectory** (Wan 2.2): specialised experts for different noise levels.

### 5.6 Connection to the Rest of the Course

| Seminar / Lecture | Image / theory side | Video analogue |
|---|---|---|
| Seminar 9 | SD 1.5 U-Net | VDM + factorized attention |
| Seminar 10 | ControlNet / IP-Adapter / LoRA | Camera control, DragNUWA, MotionDirector |
| Seminar 11 | DiT / MMDiT / FLUX | Sora, CogVideoX, HunyuanVideo, Wan |
| Lectures 11–12 | Flow Matching, Conditional FM | Every modern video model uses v-prediction / RF |
| Lectures 13–14 (discrete diffusion) | Masked diffusion language models | **MAGVIT-v2 — LLM over discrete video tokens** |

The next block of lectures will bring diffusion and language models together for us — and as MAGVIT-v2 shows, that merge is already happening on the video side.

## 6. Summary — Image Diffusion (Seminar 11) Side by Side with Video Diffusion

| Axis | Image (Seminar 11) | Video (this seminar) |
|---|---|---|
| **Backbone** | UNet → DiT → MMDiT → FLUX | 3D U-Net → DiT → MMDiT → Dual+Single stream → MoE-along-noise |
| **VAE** | 2D VAE | 3D causal VAE ($8\times8\times4$); extreme LTX 1:192 |
| **Text encoder** | CLIP → Dual CLIP → Triple (+T5) | T5-XXL **or** MLLM (llava-llama-3-8b) |
| **Objective** | DDPM $\epsilon$ → Rectified Flow v | Rectified Flow v (universal in 2024+) |
| **Positional enc.** | Learned / Fourier → RoPE | **3D RoPE** on $(t, h, w)$ |
| **Attention** | Cross-attn → Joint self-attn | Factorized space-time → Joint 3D |
| **Scale** | 860M → 12B | 1.3B (Wan-small) → 27B (Wan 2.2 MoE) |
| **Open problems** | Text rendering, consistency | **Long videos, motion control, temporal consistency, physics** |

[**Final takeaway**]

Video diffusion in 2024–2025 **recapitulates image diffusion's 2022–2023 evolution** with a ~6-month lag. The same four axes — architecture, VAE, text encoder, training objective — play out in the same order, just applied to 4D tensors instead of 3D ones.

The next seminar block covers **discrete / masked diffusion** — the frontier where autoregressive and diffusion paradigms finally merge. MAGVIT-v2's "tokenizer + LLM beats diffusion" result is the first strong signal in that direction.

---

**References** (primary papers):

1. Ho et al. "Video Diffusion Models" (2022) — arXiv:2204.03458
2. Ho et al. "Imagen Video" (2022)
3. Singer et al. "Make-A-Video" (2022)
4. Blattmann et al. "Align Your Latents / Video LDM" (2023)
5. Blattmann et al. "Stable Video Diffusion" (2023)
6. Bar-Tal et al. "Lumiere" (2024)
7. Brooks et al. "Sora technical report" — OpenAI (Feb 2024)
8. Yang et al. "CogVideoX" — arXiv:2408.06072 (ICLR 2025)
9. Kong et al. "HunyuanVideo: A Systematic Framework for Large Video Generation Model" — Tencent (Dec 2024)
10. Genmo "Mochi 1" (Oct 2024)
11. HaCohen et al. "LTX-Video: Realtime Video Latent Diffusion" — arXiv:2501.00103 (2025)
12. Wan Team "Wan 2.1" (Feb 2025), "Wan 2.2" (July 2025)
13. DeepMind "Veo 3 technical report" (2025)
14. Yu et al. "Language Model Beats Diffusion — Tokenizer is Key" (MAGVIT-v2) — arXiv:2310.05737
15. Yan et al. "VideoGPT" — arXiv:2104.10157
16. Bruce et al. "Genie" — arXiv:2402.15391
17. Valevski et al. "GameNGen: Diffusion Models Are Real-Time Game Engines" — arXiv:2408.14837
18. VideoRoPE — arXiv:2502.05173
19. Sliding Tile Attention — arXiv:2502.04507
20. Mixture of Contexts — arXiv:2508.21058